In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
import json
import typing as t
import re

In [ ]:
from src.dataset import COQGYM_TEST_SAMPLED_DATASET, COQ_WIGDERSON_TEST_SAMPLED_DATASET, COQ_BB5_TEST_DATASET, PNV_ROCQLIB_TEST_DATASET

print(f"number of CoqGym100 examples: {len(COQGYM_TEST_SAMPLED_DATASET)}")
print(f"number of Wigderson100 examples: {len(COQ_WIGDERSON_TEST_SAMPLED_DATASET)}")
print(f"number of BB5100 examples: {len(COQ_BB5_TEST_DATASET)}")
print(f"number of PnVRocqLib100 examples: {len(PNV_ROCQLIB_TEST_DATASET)}")

datasets = [
    COQGYM_TEST_SAMPLED_DATASET,
    COQ_WIGDERSON_TEST_SAMPLED_DATASET,
    COQ_BB5_TEST_DATASET,
    PNV_ROCQLIB_TEST_DATASET
]


In [ ]:
results_dirs = [
    root_directory / "evaluations/CoqGym100/Cobblestone/test_with_hammer_max_depth_5_preceding-lemmas-only-cb24f8b1e1694a86af562bd89910e083",
    root_directory / "evaluations/Wigderson100/Cobblestone/wigderson_test_with_hammer_max_depth_5_preceding-lemmas-only-849eab9ab3d04c89b9aeef60f243d133",
    root_directory / "evaluations/BB5100/Cobblestone/coq_bb5_test_with_hammer_max_depth_5_preceding-lemmas-only-4d3e6b16907043208456d33360e3cd8d",
    root_directory / "evaluations/PnVRocqLib100/Cobblestone/pnvrocqlib_test_with_hammer_max_depth_5_preceding-lemmas-only-ec66921417dc47bebac0cdac7362c78e"
]

sample_files = []
for results_dir, dataset in zip(results_dirs, datasets):
    for file in results_dir.glob("*-*.v-*.json"):
        if any(example.name == file.stem for example in dataset):
            sample_files.append(file)


# sample_fileses = [
#     list(results_dir.glob("*-*.v-*.json"))
#     for results_dir in results_dirs
# ]
# sample_files = [
#     sample_file
#     for sample_files in sample_fileses
#     for sample_file in sample_files
# ]

# should be 400
print(f"number of sample files: {len(sample_files)}")
# sample_files

In [ ]:
from src.strategy.goal_decomposition.utils import get_proof_and_num_samples

successful_proofs = {}
for sample_file in sample_files:
    try:
        with open(sample_file, "r") as f:
            data = json.load(f)
        example_name = sample_file.stem
        proof, num_samples = get_proof_and_num_samples(data)
        if proof is not None:
            successful_proofs[example_name] = (proof, num_samples)
    except Exception as e:
        print(f"Error processing {sample_file}: {e}")

# should be 173
print(f"number of proofs: {len(successful_proofs)}")


# length

In [ ]:
import numpy as np
import typing as t


def get_generator_length(generator: t.Generator[t.Any, t.Any, t.Any]) -> int:
    return sum(1 for _ in generator)

lengths = {
    example_name: get_generator_length(proof.walk())
    for example_name, (proof, _) in successful_proofs.items()
}

# statistics about the lengths
print("lengths")
print(f"mean: {np.mean(list(lengths.values()))}")
print(f"median: {np.median(list(lengths.values()))}")
print(f"max: {np.max(list(lengths.values()))}")
print(f"min: {np.min(list(lengths.values()))}")
print(f"count: {len(lengths)}")
print()

lengths_greater_than_1 = {
    example_name: length
    for example_name, length in lengths.items()
    if length > 1
}

print("lengths greater than 1")
print(f"count: {len(lengths_greater_than_1)}")
print(f"mean: {np.mean(list(lengths_greater_than_1.values()))}")
print(f"median: {np.median(list(lengths_greater_than_1.values()))}")
print(f"min: {np.min(list(lengths_greater_than_1.values()))}")
print(f"max: {np.max(list(lengths_greater_than_1.values()))}")

In [ ]:
# a histogram of the lenghs of the proofs
import matplotlib.pyplot as plt

# Create a histogram of the lengths of the proofs
plt.figure(figsize=(10, 6))
plt.hist(list(lengths.values()), bins=range(1, max(lengths.values()) + 2), edgecolor='black', align='left')
plt.xlabel('Proof Length (Number of Tactics)')
plt.ylabel('Number of Proofs')
plt.title('Histogram of Proof Lengths')
plt.xticks(range(1, max(lengths.values()) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


In [ ]:
# a histogram of lengths greater than 1
# a histogram of the lengths of the proofs with length greater than 1

plt.figure(figsize=(10, 6))
plt.hist(
    list(lengths_greater_than_1.values()),
    bins=range(2, max(lengths_greater_than_1.values()) + 2),
    edgecolor='black',
    align='left'
)
plt.xlabel('Proof Length (Number of Tactics, >1)')
plt.ylabel('Number of Proofs')
plt.title('Histogram of Proof Lengths (>1)')
plt.xticks(range(2, max(lengths_greater_than_1.values()) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


# proof contents

In [ ]:
# proofs with length greater than 1

proofs_with_length_greater_than_1 = {
    example_name: proof
    for example_name, (proof, _) in successful_proofs.items()
    if get_generator_length(proof.walk()) > 1
}

print(f"number of proofs with length greater than 1: {len(proofs_with_length_greater_than_1)}")

for example_name, proof in proofs_with_length_greater_than_1.items():
    print(f"{example_name} ({get_generator_length(proof.walk())} tactics)")
    print(proof.pretty_print())
    print()

In [ ]:
print("num containing destruct:", len([proof for proof,_ in successful_proofs.values() if any('destruct' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing induction:", len([proof for proof,_ in successful_proofs.values() if any('induction' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing split:", len([proof for proof,_ in successful_proofs.values() if any('split' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing apply:", len([proof for proof,_ in successful_proofs.values() if any('apply' in tactic.raw_text for tactic in proof.tactics)]))
print('rewrite:', len([proof for proof,_ in successful_proofs.values() if any('rewrite' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing exists:", len([proof for proof,_ in successful_proofs.values() if any('exists' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing remember:", len([proof for proof,_ in successful_proofs.values() if any('remember' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing inversion:", len([proof for proof,_ in successful_proofs.values() if any('inversion' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing rewrite:", len([proof for proof,_ in successful_proofs.values() if any('rewrite' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing ;:", len([proof for proof,_ in successful_proofs.values() if any(';' in tactic.raw_text for tactic in proof.tactics)]))
print("num containing try:", len([proof for proof,_ in successful_proofs.values() if any('try' in tactic.raw_text for tactic in proof.tactics)]))
print("generalize dependent:", len([proof for proof,_ in successful_proofs.values() if any('generalize dependent' in tactic.raw_text for tactic in proof.tactics)]))
print('remember:', len([proof for proof,_ in successful_proofs.values() if any('remember' in tactic.raw_text for tactic in proof.tactics)]))
print('assert:', len([proof for proof,_ in successful_proofs.values() if any('assert' in tactic.raw_text for tactic in proof.tactics)]))

In [ ]:
# make a big list of all the tactics, then make a histogram of the first words of the tactics

tactics = []
for proof, _ in successful_proofs.values():
    tactics.extend(proof.tactics)

# make a histogram of the first words of the tactics
plt.figure(figsize=(10, 6))
exclude_words = [
    'hammer',
    'sfirstorder',
    '-',
    'sauto',
    'hauto',
    'fcrush',
    'hecrush',
    'srun',
    'intros',
    'intro'
]
first_words = [
    tactic.raw_text.split()[0].rstrip('.;')
    for tactic in tactics
    if not any(word in tactic.raw_text for word in exclude_words)
]
from collections import Counter

# Count the frequency of each first word
first_word_counts = Counter(first_words)
# Sort the first words by frequency (descending), then alphabetically
sorted_first_words = [word for word, _ in first_word_counts.most_common()]

# Plot a bar chart instead of a histogram, since first words are categorical
plt.bar(sorted_first_words, [first_word_counts[word] for word in sorted_first_words], edgecolor='black')
plt.xlabel('First Word of Tactic')
plt.ylabel('Number of Tactics')
plt.title('Histogram of Tactics')
plt.xticks(rotation=90)
plt.show()

In [ ]:
proof_prefixes = {
    example_name: proof.prefix
    for example_name, (proof, _) in successful_proofs.items()
    if get_generator_length(proof.walk()) > 1 and proof.depth > 1
}

# print statistics for proof prefixes
print(f"number of proof prefixes: {len(proof_prefixes)}")
print(f"mean: {np.mean([len(prefix) for prefix in proof_prefixes.values()])}")
print(f"median: {np.median([len(prefix) for prefix in proof_prefixes.values()])}")
print(f"max: {max([len(prefix) for prefix in proof_prefixes.values()])}")
print(f"min: {min([len(prefix) for prefix in proof_prefixes.values()])}")

for example_name, proof_prefix in proof_prefixes.items():
    print(f"{example_name}: {len(proof_prefix)}")
    print(" ".join(tactic.pretty_print() for tactic in proof_prefix))
    print()

print(f"num containing destruct: {len([prefix for prefix in proof_prefixes.values() if any('destruct' in tactic.raw_text for tactic in prefix)])}")
print(f"num containing induction: {len([prefix for prefix in proof_prefixes.values() if any('induction' in tactic.raw_text for tactic in prefix)])}")
print(f"num containing split: {len([prefix for prefix in proof_prefixes.values() if any('split' in tactic.raw_text for tactic in prefix)])}")
print(f"num containing apply: {len([prefix for prefix in proof_prefixes.values() if any('apply' in tactic.raw_text for tactic in prefix)])}")
print(f"num containing exists: {len([prefix for prefix in proof_prefixes.values() if any('exists' in tactic.raw_text for tactic in prefix)])}")
print(f"num containing remember: {len([prefix for prefix in proof_prefixes.values() if any('remember' in tactic.raw_text for tactic in prefix)])}")

# depth

In [ ]:
depths = {
    example_name: proof.depth
    for example_name, (proof, _) in successful_proofs.items()
}

depths_greater_than_1 = {
    example_name: depth
    for example_name, depth in depths.items()
    if depth > 1
}

print("depths")
print(f"mean: {np.mean(list(depths.values()))}")
print(f"median: {np.median(list(depths.values()))}")
print(f"max: {np.max(list(depths.values()))}")
print(f"min: {np.min(list(depths.values()))}")
print(f"count: {len(depths)}")
print()

print("depths greater than 1")
print(f"mean: {np.mean(list(depths_greater_than_1.values()))}")
print(f"median: {np.median(list(depths_greater_than_1.values()))}")
print(f"max: {np.max(list(depths_greater_than_1.values()))}")
print(f"min: {np.min(list(depths_greater_than_1.values()))}")
print(f"count: {len(depths_greater_than_1)}")
print()

In [ ]:
# histogram of depths
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(list(depths.values()), bins=range(1, max(depths.values()) + 2), edgecolor='black', align='left')
plt.xlabel('Proof Depth')
plt.ylabel('Number of Proofs')
plt.title('Histogram of Proof Depths')
plt.xticks(range(1, max(depths.values()) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# a histogram of the depths of the proofs with depth greater than 1
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(list(depths_greater_than_1.values()), bins=range(2, max(depths_greater_than_1.values()) + 2), edgecolor='black', align='left')
plt.xlabel('Proof Depth (Greater than 1)')
plt.ylabel('Number of Proofs')
plt.title('Histogram of Proof Depths (Greater than 1)')
plt.xticks(range(2, max(depths_greater_than_1.values()) + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# prefixes of proofs with depth greater than 1

prefixes = {
    example_name: proof.prefix
    for example_name, (proof, _) in successful_proofs.items()
    if proof.depth > 1
}

print(f"number of prefixes: {len(prefixes)}")

for example_name, prefix in prefixes.items():
    print(f"{example_name}: {len(prefix)} in prefix, {get_generator_length(proof.walk())} tactics, depth {proof.depth}")
    print(" ".join(tactic.pretty_print() for tactic in prefix))
    print()

# num samples

In [ ]:
# histogram of num pieces (pieces = llm + hammer)

num_pieces_dict = {
    example_name: info["num_llm_samples"] + info["num_hammer_calls"]
    for example_name, (proof, info) in successful_proofs.items()
    if proof is not None
}

print(len(num_pieces_dict))

plt.figure(figsize=(10, 6))
plt.hist(list(num_pieces_dict.values()), bins=range(1, max(num_pieces_dict.values()) + 2), edgecolor='black', align='left')

# print table
for i in range(1, max(num_pieces_dict.values()) + 1):
    print(f"{i}: {len([num_pieces for num_pieces in num_pieces_dict.values() if num_pieces == i])}")

# print statistics for num pieces
print(f"Mean: {np.mean(list(num_pieces_dict.values()))}")
print(f"Median: {np.median(list(num_pieces_dict.values()))}")
print(f"Max: {max(num_pieces_dict.values())}")
print(f"Min: {min(num_pieces_dict.values())}")

In [ ]:
# histogram of num pieces s.t. num pieces > 1

num_pieces_dict = {
    example_name: info["num_llm_samples"] + info["num_hammer_calls"]
    for example_name, (proof, info) in successful_proofs.items()
    if proof is not None and info["num_llm_samples"] + info["num_hammer_calls"] > 1
}

print(len(num_pieces_dict))

plt.figure(figsize=(10, 6))
plt.hist(list(num_pieces_dict.values()), bins=range(1, max(num_pieces_dict.values()) + 2), edgecolor='black', align='left')

# math vs. software


| project name            | type     | num theorems | num theorems proven |
| ----------------------- | -------- | ------------ | ------------------- |
| coq-bb5                 | math     | 100          | 43                  |
| PnVRocqLib              | math     | 100          | 44                  |
| coq-wigderson           | software | 100          | 38                  |
| PolTac                  | math     | 3            | 3                   |
| UnifySL                 | software | 4            | 2                   |
| buchberger              | software | 13           | 4                   |
| chinese                 | math     | 2            | 2                   |
| coquelicot              | math     | 3            | 3                   |
| dblib                   | software | 4            | 0                   |
| fermat4                 | math     | 4            | 2                   |
| fundamental-arithmetics | math     | 1            | 0                   |
| goedel                  | math     | 8            | 6                   |
| huffman                 | software | 4            | 3                   |
| jordan-curve-theorem    | math     | 8            | 2                   |
| tree-automata           | software | 11           | 9                   |
| verdi                   | software | 5            | 1                   |
| verdi-raft              | software | 23           | 7                   |
| weak-up-to              | math     | 2            | 2                   |
| zfc                     | math     | 2            | 1                   |
| zorns-lemma             | math     | 3            | 1                   |






| type     | num projects | num theorems | num proven theorems | percent proven        |
| -------- | ------------ | ------------ | ------------------- | --------------------- |
| math     | 12           | 200+36       | 87+22               | 61.1% , 46.2% overall |
| software | 8            | 100+64       | 38+26               | 40.6%, 39.0% overall  |
